In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 05 (REVISED)
# The Geometry of Fragility: Feeling the Decision Boundary
# =============================================================
#
# Series:  Humble Model / Geometric Awareness
# Dataset: CIFAR-10
# Model:   CNN (3-channel input)
#
# CHANGES IN THIS REVISION:
#   1. The three geometric methods (Lipschitz, Jacobian sensitivity,
#      boundary distance) are now actually CALLED and evaluated over
#      a sample of clean and adversarial images -- the original
#      notebook defined them and never used them. Every number in
#      Section VI's table can now come from a real run.
#   2. fgsm_attack no longer calls model.train() before computing the
#      attack gradient. With 0.3 dropout in this architecture, that
#      produced a stochastic, noised gradient rather than the model's
#      actual decision surface -- not standard FGSM, and not
#      consistent with how Essays #2-4 generated attacks.
#   3. One gate-evaluation methodology, not two silently different
#      ones. This uses the Essay #4 vocabulary throughout: coverage,
#      accuracy-on-predicted, deferral rate, risk. Every method
#      (confidence-only baseline, Lipschitz, Jacobian, boundary
#      distance) is evaluated the same way, so Section VI's table is
#      an apples-to-apples comparison.
#   4. decision_gate_with_fragility is now a real fused gate (variance
#      OR confidence OR fragility), not a fragility-only stub with a
#      comment where the Essay #4 logic should be.
#   5. Sample sizes are deliberately smaller than Essay #4's 10,000-
#      image runs. Boundary distance requires up to 50 forward+backward
#      passes PER IMAGE (binary search) -- on CPU that does not scale
#      to 10k images in reasonable time. Everything below uses 300
#      images per condition, which is enough to compare methods
#      honestly, and is called out explicitly rather than silently
#      presented as equivalent to Essay #4's full-set numbers.
# =============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part B: Dataset Loading (CIFAR-10) -- unchanged
# ─────────────────────────────────────────────────────────────

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=cifar_transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=False, transform=cifar_transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

# Per-channel valid normalized range -- used below instead of a single
# scalar clamp, since CIFAR's three channels have slightly different
# mean/std and a single clamp value is only exactly correct for one.
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part C: Model Architecture -- unchanged
# ─────────────────────────────────────────────────────────────

class CNN_CIFAR10(nn.Module):
    """Simple CNN for CIFAR-10 -- 3-channel input."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part D: Baseline Model (train or load) -- unchanged
# ─────────────────────────────────────────────────────────────

def train_model(model, train_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch} -- avg loss: {avg_loss:.4f}")
    return model

def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

baseline_model = CNN_CIFAR10().to(DEVICE)

if os.path.exists('checkpoint_baseline_cifar10.pth'):
    checkpoint = torch.load('checkpoint_baseline_cifar10.pth', map_location=DEVICE)
    baseline_model.load_state_dict(checkpoint['model_state_dict'])
    print("Loaded baseline model from checkpoint")
else:
    print("Training baseline model from scratch...")
    baseline_model = train_model(baseline_model, train_loader, epochs=20)
    torch.save({'model_state_dict': baseline_model.state_dict()}, 'checkpoint_baseline_cifar10.pth')
    print("Saved baseline model checkpoint")

clean_acc = evaluate_accuracy(baseline_model, test_loader)
print(f"Baseline clean test accuracy: {clean_acc:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part E: FGSM Attack -- REVISED (no more model.train() during attack)
# ─────────────────────────────────────────────────────────────

def fgsm_attack(model, images, labels, epsilon: float = 0.03):
    """
    Generate adversarial examples using FGSM.

    FIX: the previous version called model.train() before computing
    the attack gradient. With 0.3 dropout in this architecture, that
    made the gradient direction stochastic and noised by dropout
    masking, rather than reflecting the model's actual (eval-mode)
    decision surface. Standard FGSM, and the version used in Essays
    #2-4, computes the gradient in eval mode. Fixed here -- expect
    the accuracy-drop number to shift somewhat from the previous run
    as a result; that's the noise being removed, not a new bug.
    """
    model.eval()
    images = images.clone().detach().to(DEVICE)
    images.requires_grad = True

    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels.to(DEVICE))

    model.zero_grad()
    loss.backward()

    perturbed = images + epsilon * images.grad.sign()
    perturbed = clamp_valid(perturbed)
    return perturbed.detach()

def evaluate_adversarial_accuracy(model, dataloader, epsilon: float = 0.03):
    model.eval()
    correct = 0
    total = 0
    for images, labels in dataloader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        adv_images = fgsm_attack(model, images, labels, epsilon)
        with torch.no_grad():
            outputs = model(adv_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

adv_acc = evaluate_adversarial_accuracy(baseline_model, test_loader, epsilon=0.03)
print(f"Adversarial (FGSM, eps=0.03, fixed attack) test accuracy: {adv_acc:.2f}%")
print(f"Accuracy drop: {clean_acc - adv_acc:.2f} points")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part F: Method 1 -- Local Lipschitz Estimation
# (function unchanged from the draft -- it was correct, just unused)
# ─────────────────────────────────────────────────────────────

def estimate_local_lipschitz(model, image):
    model.eval()
    image = image.clone().detach().to(DEVICE)
    image.requires_grad = True
    output = model(image)
    loss = output.norm()
    model.zero_grad()
    loss.backward()
    grad_norm = image.grad.norm().item()
    return grad_norm

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part G: Method 2 -- Jacobian-Based Sensitivity
# ─────────────────────────────────────────────────────────────

def estimate_jacobian_sensitivity(model, image, label):
    model.eval()
    image = image.clone().detach().to(DEVICE)
    image.requires_grad = True
    output = model(image)
    loss = nn.CrossEntropyLoss()(output, torch.tensor([label], device=DEVICE))
    model.zero_grad()
    loss.backward()
    grad_norm = image.grad.norm().item()
    return grad_norm

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part H: Method 3 -- Distance to Decision Boundary
# FIX: clamp_valid() instead of torch.clamp(perturbed, 0, 1), which
# was wrong for normalized inputs (valid range is NOT [0,1] once
# CIFAR normalization is applied -- that bug would have made the
# binary search clamp to a range the model never actually sees in
# practice, silently corrupting every boundary-distance estimate).
# ─────────────────────────────────────────────────────────────

def estimate_boundary_distance(model, image, label, max_iters=50):
    model.eval()
    image = image.clone().detach().to(DEVICE)
    image.requires_grad = True
    output = model(image)
    loss = nn.CrossEntropyLoss()(output, torch.tensor([label], device=DEVICE))
    model.zero_grad()
    loss.backward()
    grad = image.grad.data.sign()

    low, high = 0.0, 1.0
    for _ in range(max_iters):
        mid = (low + high) / 2
        perturbed = image + mid * grad
        perturbed = clamp_valid(perturbed)
        with torch.no_grad():
            pred = model(perturbed).argmax().item()
        if pred != label:
            high = mid
        else:
            low = mid
    return (low + high) / 2

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part I: Collect fragility scores for ALL FOUR signals
# (confidence-only baseline + the three geometric methods),
# on both clean and adversarial images -- THIS is the part that
# was missing from the previous notebook. max_iters and sample size
# are kept modest because boundary distance is expensive.
# ─────────────────────────────────────────────────────────────

N_SAMPLES = 300  # per condition (clean / adversarial). Raise if you have GPU time.

def collect_all_fragility_scores(model, loader, epsilon=0.03, n_samples=300, boundary_max_iters=30):
    """
    For n_samples clean images and the same n_samples FGSM-perturbed
    images, compute: confidence-based fragility (1 - conf), Lipschitz
    estimate, Jacobian sensitivity, and boundary distance. Also
    records whether the model's prediction was correct, which is
    what the gate evaluation below needs (matching Essay #4's
    coverage/accuracy/deferral/risk framing, not the adversarial-
    membership framing the previous notebook silently used instead).
    """
    model.eval()
    out = {
        'clean': {'confidence': [], 'lipschitz': [], 'jacobian': [], 'boundary': [], 'correct': []},
        'adversarial': {'confidence': [], 'lipschitz': [], 'jacobian': [], 'boundary': [], 'correct': []}
    }

    n_seen = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        adv_images = fgsm_attack(model, images, labels, epsilon)

        for i in range(images.size(0)):
            if n_seen >= n_samples:
                break
            img = images[i:i+1]
            adv_img = adv_images[i:i+1]
            label = labels[i].item()

            for key, x in [('clean', img), ('adversarial', adv_img)]:
                with torch.no_grad():
                    logits = model(x)
                    prob = torch.softmax(logits, dim=1)
                    conf, pred = prob.max(dim=1)
                out[key]['confidence'].append(conf.item())
                out[key]['correct'].append(int(pred.item() == label))
                out[key]['lipschitz'].append(estimate_local_lipschitz(model, x))
                out[key]['jacobian'].append(estimate_jacobian_sensitivity(model, x, label))
                out[key]['boundary'].append(estimate_boundary_distance(model, x, label, max_iters=boundary_max_iters))

            n_seen += 1
            if n_seen % 50 == 0:
                print(f"   ...processed {n_seen}/{n_samples}")
        if n_seen >= n_samples:
            break

    for key in out:
        for metric in out[key]:
            out[key][metric] = np.array(out[key][metric])
    return out

print(f"\nCollecting fragility scores for {N_SAMPLES} clean + {N_SAMPLES} adversarial images...")
print("This is slower than the confidence-only version -- boundary distance")
print("alone is up to 30 forward+backward passes per image.")
fragility_data = collect_all_fragility_scores(baseline_model, test_loader, epsilon=0.03,
                                               n_samples=N_SAMPLES, boundary_max_iters=30)

print("\nRaw signal means, clean vs adversarial:")
for metric in ['confidence', 'lipschitz', 'jacobian', 'boundary']:
    c = fragility_data['clean'][metric].mean()
    a = fragility_data['adversarial'][metric].mean()
    print(f"   {metric:12s}: clean={c:.4f}  adversarial={a:.4f}  ratio={a/c if c != 0 else float('nan'):.2f}x")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part J: Evaluate each signal as a gate, Essay-4-style
# (coverage / accuracy-on-predicted / deferral rate / risk),
# instead of the two silently-different precision/recall
# definitions the previous notebook used.
# ─────────────────────────────────────────────────────────────

def evaluate_signal_as_gate(clean_scores, clean_correct, adv_scores, adv_correct, threshold, higher_is_more_fragile=True):
    """
    A single fragility signal used as a standalone gate: defer if
    score crosses threshold. Reports coverage/accuracy/deferral/risk
    SEPARATELY for the clean set and the adversarial set, matching
    Essay #4's table structure exactly, so the two are directly
    comparable to each other and to Essay #4's own numbers.
    """
    def _metrics(scores, correct):
        if higher_is_more_fragile:
            defer_mask = scores > threshold
        else:
            defer_mask = scores < threshold
        predict_mask = ~defer_mask
        n_total = len(scores)
        n_predicted = predict_mask.sum()
        n_deferred = defer_mask.sum()
        if n_predicted > 0:
            acc = 100 * correct[predict_mask].sum() / n_predicted
        else:
            acc = float('nan')
        coverage = 100 * n_predicted / n_total
        deferral = 100 * n_deferred / n_total
        risk = 100 - acc if not np.isnan(acc) else float('nan')
        return {'accuracy': acc, 'coverage': coverage, 'deferral_rate': deferral, 'risk': risk,
                'n_predicted': int(n_predicted), 'n_deferred': int(n_deferred), 'total': n_total}

    return {
        'clean': _metrics(clean_scores, clean_correct),
        'adversarial': _metrics(adv_scores, adv_correct)
    }

# Confidence-only baseline (this IS the previous notebook's actual
# signal, correctly labeled as confidence-based rather than "no
# fragility" -- there's no zero-signal baseline in a gate, only a
# choice of which signal to gate on).
CONFIDENCE_THRESHOLD = 0.7  # matches Essay #4's confidence threshold, for comparability
baseline_result = evaluate_signal_as_gate(
    1 - fragility_data['clean']['confidence'], fragility_data['clean']['correct'],
    1 - fragility_data['adversarial']['confidence'], fragility_data['adversarial']['correct'],
    threshold=1 - CONFIDENCE_THRESHOLD, higher_is_more_fragile=True
)

# The three geometric methods -- thresholds below are placeholders at
# each signal's clean-set 75th percentile (defer the most-fragile
# quarter of clean images), a reasonable starting point for a first
# run. Revisit once you see how each distribution actually looks.
def pct75(arr):
    return float(np.percentile(arr, 75))

lipschitz_threshold = pct75(fragility_data['clean']['lipschitz'])
jacobian_threshold = pct75(fragility_data['clean']['jacobian'])
boundary_threshold = float(np.percentile(fragility_data['clean']['boundary'], 25))  # LOWER boundary distance = MORE fragile

lipschitz_result = evaluate_signal_as_gate(
    fragility_data['clean']['lipschitz'], fragility_data['clean']['correct'],
    fragility_data['adversarial']['lipschitz'], fragility_data['adversarial']['correct'],
    threshold=lipschitz_threshold, higher_is_more_fragile=True
)
jacobian_result = evaluate_signal_as_gate(
    fragility_data['clean']['jacobian'], fragility_data['clean']['correct'],
    fragility_data['adversarial']['jacobian'], fragility_data['adversarial']['correct'],
    threshold=jacobian_threshold, higher_is_more_fragile=True
)
boundary_result = evaluate_signal_as_gate(
    fragility_data['clean']['boundary'], fragility_data['clean']['correct'],
    fragility_data['adversarial']['boundary'], fragility_data['adversarial']['correct'],
    threshold=boundary_threshold, higher_is_more_fragile=False  # SMALL boundary distance = fragile
)

print("\n" + "=" * 60)
print("GATE COMPARISON -- all four signals, same evaluation")
print("=" * 60)
for name, result, thresh in [
    ("Confidence-only (threshold=0.7)", baseline_result, CONFIDENCE_THRESHOLD),
    ("Local Lipschitz", lipschitz_result, lipschitz_threshold),
    ("Jacobian Sensitivity", jacobian_result, jacobian_threshold),
    ("Boundary Distance", boundary_result, boundary_threshold),
]:
    print(f"\n{name} (threshold={thresh:.4f}):")
    print(f"   Clean:       {result['clean']}")
    print(f"   Adversarial: {result['adversarial']}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part K: Fused gate -- REAL implementation this time
# (the previous decision_gate_with_fragility only checked fragility;
# this actually combines confidence AND a geometric signal, matching
# what Section IV of the essay describes)
# ─────────────────────────────────────────────────────────────

def fused_gate(conf, geometric_score, conf_threshold=0.7, geo_threshold=None, higher_is_more_fragile=True):
    """Defer if confidence is low OR the geometric signal indicates fragility."""
    if geo_threshold is None:
        raise ValueError("geo_threshold must be set -- see the pct75/pct25 values computed above")
    low_conf = conf < conf_threshold
    if higher_is_more_fragile:
        fragile = geometric_score > geo_threshold
    else:
        fragile = geometric_score < geo_threshold
    return low_conf | fragile

def evaluate_fused_gate(clean_conf, clean_geo, clean_correct, adv_conf, adv_geo, adv_correct,
                         conf_threshold, geo_threshold, higher_is_more_fragile=True):
    def _metrics(conf, geo, correct):
        defer_mask = fused_gate(conf, geo, conf_threshold, geo_threshold, higher_is_more_fragile)
        predict_mask = ~defer_mask
        n_total = len(conf)
        n_predicted = predict_mask.sum()
        n_deferred = defer_mask.sum()
        acc = 100 * correct[predict_mask].sum() / n_predicted if n_predicted > 0 else float('nan')
        return {'accuracy': acc, 'coverage': 100 * n_predicted / n_total,
                'deferral_rate': 100 * n_deferred / n_total,
                'risk': 100 - acc if not np.isnan(acc) else float('nan')}
    return {
        'clean': _metrics(clean_conf, clean_geo, clean_correct),
        'adversarial': _metrics(adv_conf, adv_geo, adv_correct)
    }

print("\n" + "=" * 60)
print("FUSED GATE -- confidence OR boundary distance")
print("(boundary distance chosen as the geometric signal since it's")
print(" the most direct measure of the three; swap in lipschitz/")
print(" jacobian results above if either separates better)")
print("=" * 60)
fused_result = evaluate_fused_gate(
    fragility_data['clean']['confidence'], fragility_data['clean']['boundary'], fragility_data['clean']['correct'],
    fragility_data['adversarial']['confidence'], fragility_data['adversarial']['boundary'], fragility_data['adversarial']['correct'],
    conf_threshold=CONFIDENCE_THRESHOLD, geo_threshold=boundary_threshold, higher_is_more_fragile=False
)
print(f"   Clean:       {fused_result['clean']}")
print(f"   Adversarial: {fused_result['adversarial']}")
print("\nCompare against confidence-only baseline above -- the fused row's")
print("adversarial deferral rate is the number Section VI actually needs:")
print("does adding a geometric signal defer on MORE adversarial inputs")
print("than confidence alone, without destroying clean coverage the way")
print("Essay #4's naive OOD fusion did on its first two attempts?")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Part L: Visualization
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_names = ['lipschitz', 'jacobian', 'boundary']
titles = ['Local Lipschitz', 'Jacobian Sensitivity', 'Boundary Distance']

for ax, metric, title in zip(axes.flat[:3], metric_names, titles):
    ax.hist(fragility_data['clean'][metric], bins=40, alpha=0.6, label='Clean', color='green', density=True)
    ax.hist(fragility_data['adversarial'][metric], bins=40, alpha=0.6, label='Adversarial', color='red', density=True)
    ax.set_title(f'{title}: Clean vs Adversarial')
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

ax4 = axes.flat[3]
methods = ['Confidence', 'Lipschitz', 'Jacobian', 'Boundary']
adv_deferral = [
    baseline_result['adversarial']['deferral_rate'],
    lipschitz_result['adversarial']['deferral_rate'],
    jacobian_result['adversarial']['deferral_rate'],
    boundary_result['adversarial']['deferral_rate'],
]
clean_deferral = [
    baseline_result['clean']['deferral_rate'],
    lipschitz_result['clean']['deferral_rate'],
    jacobian_result['clean']['deferral_rate'],
    boundary_result['clean']['deferral_rate'],
]
x = np.arange(len(methods))
width = 0.35
ax4.bar(x - width/2, clean_deferral, width, label='Clean deferral %', color='green', alpha=0.7)
ax4.bar(x + width/2, adv_deferral, width, label='Adversarial deferral %', color='red', alpha=0.7)
ax4.set_xticks(x)
ax4.set_xticklabels(methods)
ax4.set_ylabel('Deferral Rate (%)')
ax4.set_title('Deferral Rate by Signal: Clean vs Adversarial')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('geometric_fragility_comparison.png', dpi=100)
plt.show()
print("\nSaved geometric_fragility_comparison.png")